# Django, QuerySet API

## Introduction to QuerySets
---

In this lesson, you'll learn how to **query data** from your database using Django's QuerySet API.

QuerySets are Django's way of reading data - they translate Python code into SQL queries.

<br>

When you work in Django, you don't want to write complex SQL commands (that’s the 'computer' language databases speak). A QuerySet is an intermediary that allows you to talk to the database using regular Python.

1. [QuerySet Basics](#QuerySet-Basics),
    - [The Manager Object](#The-Manager-Object),
    - [What is a QuerySet](#What-is-a-QuerySet),
2. [Retrieving Objects](#Retrieving-Objects),
    - [all() - Get Everything](#all()---Get-Everything),
    - [get() - Get a Single Object](#get()---Get-a-Single-Object),
    - [first() and last()](#first()-and-last()),
3. [Filtering Data](#Filtering-Data),
    - [filter() - Include Matching](#filter()---Include-Matching),
    - [exclude() - Exclude Matching](#exclude()---Exclude-Matching),
    - [Combining Filters](#Combining-Filters),
4. [Field Lookups](#Field-Lookups),
    - [Comparison Lookups](#Comparison-Lookups),
    - [String Lookups](#String-Lookups),
    - [Date Lookups](#Date-Lookups),
    - [Relationship Lookups](#Relationship-Lookups),
5. [QuerySet Chaining and Lazy Evaluation](#QuerySet-Chaining-and-Lazy-Evaluation),
    - [Chaining Methods](#Chaining-Methods),
    - [When QuerySets are Evaluated](#When-QuerySets-are-Evaluated),
6. [Ordering and Limiting](#Ordering-and-Limiting),
    - [order_by()](#order_by()),
    - [Slicing QuerySets](#Slicing-QuerySets),
7. [🧠 EXERCISE 🧠, Query the Bookstore](#🧠-EXERCISE-🧠,-Query-the-Bookstore).

<br>

## QuerySet Basics

---

### The Manager Object

---

Every Django model has a **Manager** - an interface for database operations.

By default, Django adds a manager called `objects` to every model:

<br>

```python
class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

    # IntegerChoices example used in this lesson
    category_type = models.PositiveSmallIntegerField(
        choices=CategoryType.choices,
        default=CategoryType.TECH,
    )
    # ...
```

# Access the manager via .objects
```bash
Blog.objects  # This is the manager
```

<br>

The manager is your gateway to the database - you use it to:
- **Create** objects: `Blog.objects.create(...)`
- **Read** objects: `Blog.objects.all()`, `Blog.objects.filter(...)`
- **Update** objects: `Blog.objects.filter(...).update(...)`
- **Delete** objects: `Blog.objects.filter(...).delete()`

**Quick shell demo (`filter().update()`):**

```bash
python manage.py shell
```

```python
from datetime import date
from blog.models import Blog

# 1) Create one record
blog = Blog.objects.create(
    title="ORM update demo",
    author="Shell User",
    author_email="shell@example.com",
    slug="orm-update-demo",
    content="Initial content",
    published_date=date.today(),
)

# 2) Update it using filter().update()
rows = Blog.objects.filter(id=blog.id).update(
    title="ORM update demo (updated)",
    content="Updated via filter().update()",
)
print(rows)  # 1

# 3) Verify the change
Blog.objects.get(id=blog.id).title
```

⚠️ `update()` performs a direct SQL update, so it does **not** call model `save()` and does **not** trigger `pre_save`/`post_save` signals.

<br>

### What is a QuerySet

---

A **QuerySet** represents a collection of objects from your database.

Think of it as a **lazy SQL query** - it describes what you want, but doesn't fetch data until you actually need it.

<br>

| Concept | Description |
|---------|-------------|
| **QuerySet** | A collection of database objects |
| **Lazy** | Query isn't executed until data is needed |
| **Chainable** | Methods return new QuerySets |
| **Iterable** | Can loop over results |

### Example: QuerySet basics (conceptual)
```python
from blog.models import Blog

# This creates a QuerySet but doesn't hit the database yet
blogs = Blog.objects.all()
# Type: <QuerySet [...]>

# The database is queried when you:
# 1. Iterate over the QuerySet
for blog in blogs:
    print(blog.title)

# 2. Slice the QuerySet
first_five = blogs[:5]

# 3. Convert to list
blog_list = list(blogs)

# 4. Check existence
if blogs:
    print("We have blogs!")
```

<br>

## Retrieving Objects

---

### `all()` - Get Everything

---

The `all()` method returns a QuerySet containing all objects:

```python
# Get all blogs
all_blogs = Blog.objects.all()
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog;
```

### Example: Using `all()` in the Django shell
```python
# Get all blogs
blogs = Blog.objects.all()
print(f"Total blogs: {blogs.count()}")

# Iterate over all blogs
for blog in blogs:
    print(f"- {blog.title} by {blog.author}")
```

<br>

### `get()` - Get a Single Object

---

The `get()` method retrieves **exactly one** object matching the criteria:

```python
# Get blog by primary key
blog = Blog.objects.get(pk=1)

# Get blog by unique field (slug)
blog = Blog.objects.get(slug='django-queryset-guide')
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog WHERE id = 1;
```

<br>

**⚠️ Important:** `get()` raises exceptions if:

| Exception | When |
|-----------|------|
| `DoesNotExist` | No object matches |
| `MultipleObjectsReturned` | More than one object matches |

### Example: Handling `get()` exceptions
```python
from django.core.exceptions import ObjectDoesNotExist

# Method 1: Try/except
try:
    blog = Blog.objects.get(pk=999)
except Blog.DoesNotExist:
    print("Blog not found!")

# Method 2: Use get_or_create (creates if not found)
blog, created = Blog.objects.get_or_create(
    slug='django-best-practices',
    defaults={
        'title': 'Django Best Practices',
        'author': 'John Doe',
        'published_date': '2024-01-15'
    }
)
if created:
    print("New blog created!")
else:
    print("Blog already existed.")
```

<br>

### `first()` and `last()`

---

Use `first()` and `last()` when you want one object but don't want exceptions:

```python
# Returns the first object, or None if empty
blog = Blog.objects.first()

# Returns the last object, or None if empty
blog = Blog.objects.last()

# Can be combined with filter
newest = Blog.objects.order_by('-published_date').first()
```
The difference lies in the ordering, not in `first()`/ `last()` themselves.

```
newest = Blog.objects.order_by('-published_date').first()
```
..returns the first row based on only one key: published_date DESC.

```
Blog.objects.last()
```
..uses the model's default ordering from Meta.ordering (in your case ['-published_date', 'title']) and picks the last item in that sequence.

This is why, if multiple records have the same `published_date`, the results may differ:
`order_by('-published_date')` has an undefined order when dates match (it's DB-dependent).

The default ordering includes title as a tie-breaker.

### Example: Using `first()` safely
```python
# Get the newest blog (no exception if empty)
newest_blog = Blog.objects.order_by('-published_date').first()

if newest_blog:
    print(f"Newest: {newest_blog.title}")
else:
    print("No blogs in database.")
```

<br>

## Filtering Data

---

### `filter()` - Include Matching

---

The `filter()` method returns a QuerySet with objects matching the given conditions:

```python
# Blogs by a specific author
author_blogs = Blog.objects.filter(author='John Doe')

# Blogs published in 2024
recent_blogs = Blog.objects.filter(published_date__year=2024)
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog WHERE author = 'John Doe';
```

In [ ]:
# Example: Multiple filter conditions (AND)

# Both conditions must be true (AND)
tech_blogs_2024 = Blog.objects.filter(
    category_type=1,  # Technology
    published_date__year=2024
)

# Same as:
tech_blogs_2024 = Blog.objects.filter(category_type=1).filter(published_date__year=2024)

<br>

### `exclude()` - Exclude Matching

---

The `exclude()` method returns objects that **don't** match the conditions:

```python
# All blogs except Technology category
non_tech_blogs = Blog.objects.exclude(category_type=1)

# Blogs not by a specific author
other_authors = Blog.objects.exclude(author='Unknown')
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog WHERE NOT category_type = 1;
```

In [ ]:
# Example: Combining filter and exclude

# Technology blogs with content that aren't from 2023
recent_tech_blogs = (
    Blog.objects
    .filter(title__icontains='django')   # Title contains 'django'
    .filter(category_type=1)             # Technology category
    .exclude(published_date__year=2023)  # Not from 2023
)

<br>

### Combining Filters

---

🔑 For **OR** conditions, use the `Q` object:

```python
from django.db.models import Q

# Blogs by author A OR author B
blogs = Blog.objects.filter(
    Q(author='John Doe') | Q(author='Jane Smith')
)

# Complex conditions
blogs = Blog.objects.filter(
    Q(category_type=1) | Q(category_type=4),  # Tech OR News
    published_date__year=2024  # AND published in 2024
)
```

In [ ]:
# Example: Q objects for complex queries

from django.db.models import Q

# Find blogs that are:
# (Technology AND from 2024) OR (Business category)
featured = Blog.objects.filter(
    (Q(category_type=1) & Q(published_date__year=2024)) | Q(category_type=2)
)

# Q object operators:
# | = OR
# & = AND
# ~ = NOT

# Blogs NOT by unknown authors
known_authors = Blog.objects.filter(~Q(author='Unknown'))

<br>

## Field Lookups

---

🔑 **Field lookups** are how you specify conditions in filters. They use double-underscore syntax: `field__lookup=value`

<br>

Field Lookups are suffixes you add to field names to specify to the database how to compare data (whether it should contain something, be greater, smaller, or start with a specific letter).

### Comparison Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `exact` | Exact match (default) | `name__exact='Python'` |
| `iexact` | Case-insensitive exact | `name__iexact='python'` |
| `gt` | Greater than | `price__gt=20` |
| `gte` | Greater than or equal | `price__gte=20` |
| `lt` | Less than | `price__lt=20` |
| `lte` | Less than or equal | `price__lte=20` |
| `in` | In a list | `status__in=['draft', 'review']` |
| `range` | Between values | `price__range=(10, 50)` |

In [ ]:
# Example: Comparison lookups
from blog.models import Blog

# Blogs published after a specific date
recent = Blog.objects.filter(published_date__gt='2024-01-01')

# Blogs from a range of dates
q1_blogs = Blog.objects.filter(published_date__range=('2024-01-01', '2024-03-31'))

# Blogs in specific categories
tech_or_news = Blog.objects.filter(category_type__in=[1, 4])

# Blogs published in 2024 or later
recent_blogs = Blog.objects.filter(published_date__year__gte=2024)

<br>

### String Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `contains` | Contains substring | `title__contains='Python'` |
| `icontains` | Case-insensitive contains | `title__icontains='python'` |
| `startswith` | Starts with | `title__startswith='The'` |
| `istartswith` | Case-insensitive starts with | `title__istartswith='the'` |
| `endswith` | Ends with | `title__endswith='Guide'` |
| `iendswith` | Case-insensitive ends with | `title__iendswith='guide'` |
| `regex` | Regular expression | `title__regex=r'^[A-Z]'` |
| `iregex` | Case-insensitive regex | `title__iregex=r'^the'` |

In [ ]:
# Example: String lookups

# Blogs with 'django' in title (case-insensitive)
django_blogs = Blog.objects.filter(title__icontains='django')
# Matches: "Django for Beginners", "Two Scoops of Django", "DJANGO MASTER"

# Blogs starting with "How"
how_blogs = Blog.objects.filter(title__startswith='How')
# Matches: "How to Deploy Django", "How Django Works"

# Blogs by authors with gmail addresses
gmail_authors = Blog.objects.filter(author_email__iendswith='@gmail.com')

<br>

### Date Lookups

---

| Lookup | Description | Example |
|--------|-------------|--------|
| `date` | Exact date | `created_at__date=date(2024, 3, 15)` |
| `year` | Year component | `publication_date__year=2024` |
| `month` | Month component | `publication_date__month=12` |
| `day` | Day component | `publication_date__day=25` |
| `week` | Week number | `created_at__week=1` |
| `week_day` | Day of week (1=Sunday) | `created_at__week_day=2` |

In [ ]:
# Example: Date lookups
from datetime import date, timedelta

from blog.models import Blog

# Blogs published in 2024
blogs_2024 = Blog.objects.filter(published_date__year=2024)

# Blogs published in December
december_blogs = Blog.objects.filter(published_date__month=12)

# Blogs published in the last 30 days
thirty_days_ago = date.today() - timedelta(days=30)
recent_blogs = Blog.objects.filter(published_date__gte=thirty_days_ago)

# Comments created today
from django.utils import timezone
today_comments = Comment.objects.filter(created_at__date=timezone.now().date())

<br>

### Relationship Lookups

---

🔑 Relationship lookups allow you to filter data based on information stored in other (related) tables. All you need is the relationship name and double underscores

<br>

You can traverse relationships using double underscores:
**SQL equivalent (for the first filter):**
```sql
SELECT b.*
FROM blog_blog b
JOIN blog_reviews r ON b.id = r.blog_id
JOIN auth_user u ON r.user_id = u.id
WHERE u.username = 'john_doe';
```

In [ ]:
from datetime import date
from django.contrib.auth import get_user_model

from blog.models import Blog, BlogReview

User = get_user_model()

# 1) Create deterministic test data for relationship lookups
john_doe, _ = User.objects.get_or_create(
    username='john_doe',
    defaults={'email': 'john_doe@example.com'},
)

sample_blog, _ = Blog.objects.get_or_create(
    slug='relationship-lookups-demo',
    defaults={
        'title': 'Relationship lookups demo',
        'author': 'Lesson Bot',
        'author_email': 'lesson.bot@example.com',
        'content': 'Seed blog for relationship lookup examples.',
        'published_date': date.today(),
    },
)

BlogReview.objects.update_or_create(
    blog=sample_blog,
    user=john_doe,
    defaults={
        'rating': 5,
        'comment': 'Excellent post for queryset examples.',
    },
)

# 2) Relationship filters
blogs_reviewed_by_john = Blog.objects.filter(
    reviews__user__username='john_doe'
).distinct()

blogs_with_five_star_review = Blog.objects.filter(
    reviews__rating=5
).distinct()

In [ ]:
# Example: Spanning relationships

# Blogs with comments containing specific text
commented_blogs = Blog.objects.filter(comments__content__icontains='excellent')

# Blogs reviewed by a specific user
user_reviewed = Blog.objects.filter(reviews__user__username='john_doe')

# Comments on Technology blogs
tech_comments = Comment.objects.filter(blog__category_type=1)

# Blogs that have reviews (reverse relationship)
reviewed_blogs = Blog.objects.filter(reviews__isnull=False).distinct()

<br>

| Lookup | Description |
|--------|-------------|
| `isnull` | Check for NULL | `author_email__isnull=True` |
| Spanning FK | Access related field | `comments__content='Great'` |
| Reverse FK | Access reverse relation | `reviews__rating=5` |

<br>

## QuerySet Chaining and Lazy Evaluation

---

### Chaining Methods

---

🔑 the process of stacking commands one after another using a dot (.). It allows you to create highly precise data queries by gradually adding more and more conditions into a single "chain."

Most QuerySet methods return a **new QuerySet**, allowing you to chain them:

```python
# Build up a complex query step by step
blogs = (
    Blog.objects
    .filter(category_type=1)  # Technology
    .filter(published_date__year=2024)
    .exclude(author='Unknown')
    .order_by('-published_date')
    [:10]  # First 10 results
)
```

Each method creates a new QuerySet without modifying the original!

In [ ]:
# Example: Building queries dynamically
from blog.models import Blog


def search_blogs(title=None, author=None, category_type=None, year=None):
    """Search blogs with optional filters."""
    queryset = Blog.objects.all()
    
    if title:
        queryset = queryset.filter(title__icontains=title)
    
    if author:
        queryset = queryset.filter(author__icontains=author)
    
    if category_type:
        queryset = queryset.filter(category_type=category_type)
    
    if year:
        queryset = queryset.filter(published_date__year=year)
    
    return queryset.order_by('-published_date')

# Usage:
# results = search_blogs(title='django', category_type=1, year=2026)

<br>

### When QuerySets are Evaluated

---

QuerySets are **lazy** - they don't hit the database until you actually need the data.

<br>

The statement below won't run the evaluation process:
```python
blogs = Blog.objects.all()
```

<br>

**QuerySets are evaluated when you:**

| Action | Example |
|--------|--------|
| Iterate | `for book in books:` |
| Slice with step | `books[::2]` |
| Convert to list | `list(books)` |
| Check bool | `if books:` |
| Call `len()` | `len(books)` |
| Call `repr()` | `print(books)` |
| Pickle | `pickle.dumps(books)` |

In [ ]:
# Example: Understanding lazy evaluation

# This does NOT hit the database
blogs = Blog.objects.filter(category_type=1)
blogs = blogs.filter(published_date__year=2024)
blogs = blogs.order_by('title')
# Still no database query!

# NOW the database is queried (iteration)
for blog in blogs:
    print(blog.title)

# Benefit: You can build complex queries without performance penalty
# Only ONE SQL query is executed, no matter how many filter() calls

<br>

**Caching:** Once evaluated, results are cached in the QuerySet:

```python
# First iteration - database query
for blog in blogs:
    print(blog.title)

# Second iteration - uses cached results, no database query
for blog in blogs:
    print(blog.author)
```

<br>

## Ordering and Limiting

---

### `order_by()`

---

The `order_by()` method sorts results:

```python
# Ascending (default)
blogs = Blog.objects.order_by('title')

# Descending (prefix with -)
blogs = Blog.objects.order_by('-published_date')

# Multiple fields
blogs = Blog.objects.order_by('author', '-published_date')
# First by author (A-Z), then by date (newest first)
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog ORDER BY title ASC;
SELECT * FROM blog_blog ORDER BY published_date DESC;
```

In [ ]:
# Example: Various ordering options

# Alphabetical by title
alphabetical = Blog.objects.order_by('title')

# Newest first
newest_first = Blog.objects.order_by('-published_date')

# Oldest first
oldest_first = Blog.objects.order_by('published_date')

# By author name, then by date
by_author = Blog.objects.order_by('author', '-published_date')

# Random order
random_blogs = Blog.objects.order_by('?')  # Note: Can be slow on large tables!

# Clear existing ordering (from Meta class)
unordered = Blog.objects.order_by()

<br>

### Slicing QuerySets

---

You can slice QuerySets like Python lists to limit results:

```python
# First 5 blogs
first_five = Blog.objects.all()[:5]

# Blogs 5-10 (offset 5, limit 5)
next_five = Blog.objects.all()[5:10]

# Single item (returns object, not QuerySet)
sixth_blog = Blog.objects.all()[5]
```

**SQL equivalent:**
```sql
SELECT * FROM blog_blog LIMIT 5;
SELECT * FROM blog_blog LIMIT 5 OFFSET 5;
```

In [ ]:
# Example: Pagination with slicing

def get_page(page_number: int, page_size: int = 10):
    """Get a page of blogs."""
    start = (page_number - 1) * page_size
    end = start + page_size
    return Blog.objects.order_by('-published_date')[start:end]

# Usage:
# page_1 = get_page(1)  # Blogs 0-9
# page_2 = get_page(2)  # Blogs 10-19
# page_3 = get_page(3)  # Blogs 20-29

<br>

🔑 **Note:** Negative indexing is NOT supported:

```python
# This will raise an error!
last_blog = Blog.objects.all()[-1]  # AssertionError

# Use this instead:
last_blog = Blog.objects.order_by('-id').first()
# or
last_blog = Blog.objects.last()
```

<br>

## Useful QuerySet Methods

---

Here are some additional methods you'll use frequently:

In [ ]:
# count() - Get the count without loading all objects
blog_count = Blog.objects.filter(category_type=1).count()
# SQL: SELECT COUNT(*) FROM blog_blog WHERE category_type = 1;

# exists() - Check if any objects match (efficient)
has_tech_blogs = Blog.objects.filter(category_type=1).exists()
# Returns True/False without loading objects

# values() - Get dictionaries instead of model instances
titles = Blog.objects.values('title', 'author')
# Returns: [{'title': 'Blog 1', 'author': 'John'}, ...]

# values_list() - Get tuples instead of dictionaries
titles = Blog.objects.values_list('title', flat=True)
# Returns: ['Blog 1', 'Blog 2', 'Blog 3', ...]

# distinct() - Remove duplicates
unique_authors = Blog.objects.values('author').distinct()

<br>

## Summary

---

| Method | Description |
|--------|-------------|
| `all()` | Get all objects |
| `get()` | Get single object (raises exception if not found) |
| `first()` / `last()` | Get first/last object (or None) |
| `filter()` | Include objects matching conditions |
| `exclude()` | Exclude objects matching conditions |
| `order_by()` | Sort results |
| `count()` | Count objects efficiently |
| `exists()` | Check if any objects exist |
| `values()` | Return dictionaries instead of objects |

<br>

**Common lookups:**

| Lookup | Example |
|--------|--------|
| `__exact` | `name__exact='value'` (default) |
| `__icontains` | `title__icontains='python'` |
| `__gt`, `__lt` | `price__gt=20`, `price__lt=50` |
| `__in` | `status__in=['draft', 'review']` |
| `__isnull` | `author__isnull=True` |
| `__year`, `__month` | `date__year=2024` |

<br>

### 🧠 EXERCISE 🧠, Query the Blog

---

Using the Django shell, practice QuerySet operations:

1. Get all Technology blogs
2. Get the 5 most recent blogs
3. Find blogs with "Django" in the title (case-insensitive)
4. Get blogs published in the last year
5. Find blogs by authors whose name contains "John"
6. Count how many blogs have comments
7. Get a list of unique author names (use `values_list`)
8. Find blogs that are either Technology OR News category

<details>
    <summary>▶️ Solution</summary>
    
```python
# Start Django shell
# python manage.py shell

from blog.models import Blog
from django.db.models import Q
from datetime import date, timedelta

# 1. All Technology blogs
tech_blogs = Blog.objects.filter(category_type=1)

# 2. 5 most recent blogs
recent = Blog.objects.order_by('-published_date')[:5]

# 3. Blogs with "Django" in title
django_blogs = Blog.objects.filter(title__icontains='django')

# 4. Blogs published in last year
one_year_ago = date.today() - timedelta(days=365)
recent_blogs = Blog.objects.filter(published_date__gte=one_year_ago)

# 5. Blogs by authors containing "John"
john_blogs = Blog.objects.filter(author__icontains='john')

# 6. Count blogs with comments
commented_count = Blog.objects.filter(comments__isnull=False).distinct().count()

# 7. Unique author names
authors = Blog.objects.values_list('author', flat=True).distinct()

# 8. Technology OR News category
tech_or_news = Blog.objects.filter(
    Q(category_type=1) | Q(category_type=4)
)
```
</details>

<br>

### 🧠 EXERCISE 🧠, Build a Blog Search Function

---

Create a search function for blogs that accepts optional parameters:

1. `query` - search in title and content (case-insensitive)
2. `author` - filter by author name
3. `category_type` - filter by category type (1=Tech, 2=Business, 3=Lifestyle, 4=News)
4. `year` - filter by publication year
5. `has_comments` - only show blogs with comments

The function should:
- Only apply filters for provided parameters
- Order results by relevance (title match first) then by publication date
- Return distinct results to avoid duplicates from joins

<details>
    <summary>▶️ Solution</summary>
    
```python
from django.db.models import Q, Case, When, Value, IntegerField

def search_blogs(
    query: str = None,
    author: str = None,
    category_type: int = None,
    year: int = None,
    has_comments: bool = False
):
    """Search blogs with optional filters."""
    # Start with all blogs
    qs = Blog.objects.all()
    
    # Search query in title and content
    if query:
        qs = qs.filter(
            Q(title__icontains=query) | Q(content__icontains=query)
        )
    
    # Author filter
    if author:
        qs = qs.filter(author__icontains=author)
    
    # Category filter
    if category_type is not None:
        qs = qs.filter(category_type=category_type)
    
    # Year filter
    if year:
        qs = qs.filter(published_date__year=year)
    
    # Has comments filter
    if has_comments:
        qs = qs.filter(comments__isnull=False).distinct()
    
    # Order by title match (if query) then by date
    if query:
        qs = qs.annotate(
            title_match=Case(
                When(title__icontains=query, then=Value(0)),
                default=Value(1),
                output_field=IntegerField()
            )
        ).order_by('title_match', '-published_date')
    else:
        qs = qs.order_by('-published_date')
    
    return qs.distinct()

# Usage:
# results = search_blogs(query='django', category_type=1, year=2024)
```
</details>

---